In [1]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np

In [2]:
# 호텔 예약 데이터에서 필요한 날짜 추출
# 환율 데이터는 예약한 날짜와 실제 호텔 투숙 날짜의 차이가 유의미한 데이터 이므로 호텔 예약 데이터에서 예약 날짜와 도착 날짜 모두 필요
df_hotel = pd.read_csv('../data/preprocessed/hotel_bookings.csv')

r_date_min = df_hotel['reservation_date'].min()
r_date_max = df_hotel['reservation_date'].max()

a_date_min = df_hotel['arrival_date'].min()
a_date_max = df_hotel['arrival_date'].max()

print(f'타겟 날짜: {min(r_date_min, a_date_min)} ~ {max(r_date_max, a_date_max)}')

타겟 날짜: 2013-06-24 ~ 2017-08-31


In [3]:
# 호텔 예약 데이터에서 필요한 국가 추출
df_hotel['country'].unique()

<StringArray>
['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA', 'ROU', 'NOR', 'OMN', 'ARG',
 ...
 'ATA', 'GTM', 'ASM', 'MRT', 'NCL', 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 177, dtype: str

In [4]:
# 타겟 국가 줄이기
# 호텔 예약 데이터에서 국가별 최빈값 순으로 새로운 데이터프레임 생성
df_country = df_hotel['country'].value_counts().reset_index()
df_country

,country,count
0,PRT,48590
1,GBR,12129
2,FRA,10415
3,ESP,8568
4,DEU,7287
...,...,...
172,NCL,1
173,KIR,1
174,SDN,1
175,ATF,1


In [ ]:
# count 100 이상인 데이터로 추리기
df_country = df_country.loc[df_country['count'] > 100]
df_country

,country,count,percentage
0,PRT,48590,41.739683
1,GBR,12129,10.419029
2,FRA,10415,8.946672
3,ESP,8568,7.360066
4,DEU,7287,6.259664
5,ITA,3766,3.235062
6,IRL,3375,2.899186
7,BEL,2342,2.011820
8,BRA,2224,1.910456
9,NLD,2104,1.807374


In [6]:
df_country['country'].unique()

<StringArray>
['PRT', 'GBR', 'FRA', 'ESP', 'DEU', 'ITA', 'IRL', 'BEL', 'BRA', 'NLD', 'USA',
 'CHE',  'CN', 'AUT', 'SWE', 'CHN', 'POL', 'ISR', 'RUS', 'NOR', 'ROU', 'FIN',
 'DNK', 'AUS', 'AGO', 'LUX', 'MAR', 'TUR', 'HUN', 'ARG', 'JPN', 'CZE', 'IND',
 'KOR', 'GRC', 'DZA', 'SRB']
Length: 37, dtype: str

In [23]:
# 각 국가의 화폐단위 추가
# 데이터 불러오기
df_currency = pd.read_csv('../data/raw/country_currencies.csv')
df_currency.head()

,country_code - currency
0,PRT: 포르투갈 — 유로 (EUR)
1,FRA: 프랑스 — 유로 (EUR)
2,ESP: 스페인 — 유로 (EUR)
3,DEU: 독일 — 유로 (EUR)
4,ITA: 이탈리아 — 유로 (EUR)


In [24]:
# 정규식을 이용해 4개의 열로 분할
pattern = r'(?P<country_code>\w+):\s*(?P<country>[가-힣]+)\s*—\s*(?P<currency>[가-힣]+)\s*\((?P<currency_code>\w+)\)'

# 추출 후 원본 데이터프레임과 가로로 합치기(concat)
df_currency = df_currency['country_code - currency'].str.extract(pattern)
df_currency

,country_code,country,currency,currency_code
0,PRT,포르투갈,유로,EUR
1,FRA,프랑스,유로,EUR
2,ESP,스페인,유로,EUR
3,DEU,독일,유로,EUR
4,ITA,이탈리아,유로,EUR
5,IRL,아일랜드,유로,EUR
6,BEL,벨기에,유로,EUR
7,NLD,네덜란드,유로,EUR
8,AUT,오스트리아,유로,EUR
9,FIN,핀란드,유로,EUR


In [26]:
# df_country에 df_currency 합치기
df = pd.merge(df_country, df_currency, how='inner', left_on='country', right_on='country_code')
df.head()

,country_x,count,percentage,country_code,country_y,currency,currency_code
0,PRT,48590,41.739683,PRT,포르투갈,유로,EUR
1,FRA,10415,8.946672,FRA,프랑스,유로,EUR
2,ESP,8568,7.360066,ESP,스페인,유로,EUR
3,DEU,7287,6.259664,DEU,독일,유로,EUR
4,ITA,3766,3.235062,ITA,이탈리아,유로,EUR


In [ ]:
# 중복 열 제거 및 순서 변경
df = df[['country_code', 'count', 'country_y', 'currency', 'currency_code']]
df.rename(columns={'country_y': 'country'}, inplace=True)
df

,country_code,count,country,currency,currency_code
0,PRT,48590,포르투갈,유로,EUR
1,FRA,10415,프랑스,유로,EUR
2,ESP,8568,스페인,유로,EUR
3,DEU,7287,독일,유로,EUR
4,ITA,3766,이탈리아,유로,EUR
5,IRL,3375,아일랜드,유로,EUR
6,BEL,2342,벨기에,유로,EUR
7,BRA,2224,브라질,헤알,BRL
8,NLD,2104,네덜란드,유로,EUR
9,CN,1279,중국,위안,CNY


In [42]:
df_currency_cnt = df.groupby('currency')[['count']].sum()
df_currency_cnt.sort_values(by='count', ascending=False, inplace=True)
df_currency_cnt.reset_index(inplace=True)
df_currency_cnt['currency %'] = df_currency_cnt['count'] / df_currency_cnt['count'].sum() * 100
df_currency_cnt

,currency,count,currency %
0,유로,88572,89.141615
1,위안,2278,2.292650
2,헤알,2224,2.238303
3,크로네,1042,1.048701
4,크로나,1024,1.030585
5,즈워티,919,0.924910
6,루블,632,0.636064
7,레우,500,0.503216
8,콴자,362,0.364328
9,디르함,259,0.260666


In [7]:
# 환율 데이터 불러오기
df = pd.read_csv('../data/raw/euro-daily-hist_1999_2022.csv')
df.head()

,Period\Unit:,[Australian dollar ],[Bulgarian lev ],[Brazilian real ],[Canadian dollar ],[Swiss franc ],[Chinese yuan renminbi ],[Cypriot pound ],[Czech koruna ],[Danish krone ],...,[Romanian leu ],[Russian rouble ],[Swedish krona ],[Singapore dollar ],[Slovenian tolar ],[Slovak koruna ],[Thai baht ],[Turkish lira ],[US dollar ],[South African rand ]
0,2025-04-02,1.7146,1.9558,6.1212,1.5479,0.9543,7.8529,NaN,24.963,7.4611,...,4.9775,NaN,10.764,1.4508,NaN,NaN,36.93,40.9573,1.0803,20.1042
1,2025-04-01,1.7255,1.9558,6.1679,1.5529,0.952,7.8431,NaN,24.954,7.4616,...,4.9774,NaN,10.816,1.4492,NaN,NaN,36.846,40.9201,1.0788,19.7741
2,2025-03-31,1.7318,1.9558,6.2507,1.5533,0.9531,7.8442,NaN,24.962,7.4613,...,4.9771,NaN,10.849,1.4519,NaN,NaN,36.706,41.0399,1.0815,19.8782
3,2025-03-28,1.712,1.9558,6.2252,1.5444,0.9525,7.8445,NaN,24.96,7.4616,...,4.9774,NaN,10.82,1.4481,NaN,NaN,36.664,41.0387,1.0797,19.6113
4,2025-03-27,1.7101,1.9558,6.2154,1.5425,0.9524,7.8361,NaN,24.982,7.4605,...,4.9773,NaN,10.8235,1.445,NaN,NaN,36.529,40.9940,1.0785,19.7061


In [8]:
# 데이터 정보
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6785 entries, 0 to 6784
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Period\Unit:              6785 non-null   str    
 1   [Australian dollar ]      6785 non-null   str    
 2   [Bulgarian lev ]          6383 non-null   str    
 3   [Brazilian real ]         6517 non-null   str    
 4   [Canadian dollar ]        6785 non-null   str    
 5   [Swiss franc ]            6785 non-null   str    
 6   [Chinese yuan renminbi ]  6517 non-null   str    
 7   [Cypriot pound ]          2346 non-null   str    
 8   [Czech koruna ]           6785 non-null   str    
 9   [Danish krone ]           6785 non-null   str    
 10  [Estonian kroon ]         3130 non-null   str    
 11  [UK pound sterling ]      6785 non-null   str    
 12  [Greek drachma ]          520 non-null    str    
 13  [Hong Kong dollar ]       6785 non-null   str    
 14  [Croatian kuna ]   

In [9]:
# 날짜 데이터 타입 변경
df['date'] = pd.to_datetime(df[r'Period\Unit:'], format='%Y-%m-%d')
df['date']

0      2025-04-02
1      2025-04-01
2      2025-03-31
3      2025-03-28
4      2025-03-27
          ...    
6780   1999-01-08
6781   1999-01-07
6782   1999-01-06
6783   1999-01-05
6784   1999-01-04
Name: date, Length: 6785, dtype: datetime64[us]

In [10]:
df.describe()

,[Iceland krona ],[Romanian leu ],[Turkish lira ],date
count,4378.000000,6723.000000,6723.000000,6785
mean,112.368314,4.042081,6.019502,2012-01-26 00:47:06.941783
min,68.070000,1.291200,0.370100,1999-01-04 00:00:00
25%,84.622500,3.597700,1.750450,2005-07-05 00:00:00
50%,92.495000,4.350500,2.330700,2012-01-04 00:00:00
75%,144.900000,4.661500,6.022200,2018-08-20 00:00:00
max,305.000000,4.978300,41.399700,2025-04-02 00:00:00
std,34.715841,0.883798,8.784625,NaN


In [11]:
# 결측치 확인
df.isna().sum()

Period\Unit:                   0
[Australian dollar ]           0
[Bulgarian lev ]             402
[Brazilian real ]            268
[Canadian dollar ]             0
[Swiss franc ]                 0
[Chinese yuan renminbi ]     268
[Cypriot pound ]            4439
[Czech koruna ]                0
[Danish krone ]                0
[Estonian kroon ]           3655
[UK pound sterling ]           0
[Greek drachma ]            6265
[Hong Kong dollar ]            0
[Croatian kuna ]             844
[Hungarian forint ]            0
[Indonesian rupiah ]           0
[Israeli shekel ]            268
[Indian rupee ]              268
[Iceland krona ]            2407
[Japanese yen ]                0
[Korean won ]                  0
[Lithuanian litas ]         2626
[Latvian lats ]             2881
[Maltese lira ]             4439
[Mexican peso ]                0
[Malaysian ringgit ]           0
[Norwegian krone ]             0
[New Zealand dollar ]          0
[Philippine peso ]             0
[Polish zl